# FOLIO Syllogism Multi-Label Graph Classification Pipeline
### Backbone: Google Flan-T5-Large
This notebook contains the complete Neurosymbolic Graph pipeline for classifying the FOLIO syllogism dataset using PyTorch Geometric, NetworkX, and Hugging Face Transformers. It includes automated contrapositive data augmentation.

In [1]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install PyTorch Geometric and dependencies
!pip install torch-geometric

import os
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import networkx as nx
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import RGCNConv, GlobalAttention

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.0 MB/s  0:00:00
Using device: cuda


## 1. Relational Logic GNN Core Architecture (Flan-T5-Large Backend)

In [21]:
from torch_geometric.nn.aggr import AttentionalAggregation

class LogicGraphClassifier(nn.Module):
    def __init__(self, pretrained_model_name="google/flan-t5-large", num_relations=2, hidden_dim=256, num_classes=3):
        super(LogicGraphClassifier, self).__init__()

        # Initialize Flan-T5 Large Tokenizer and Encoder
        self.tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name)
        t5_model = AutoModel.from_pretrained(pretrained_model_name)
        self.text_encoder = t5_model.encoder

        # Freeze all backbone parameters completely to guarantee training stability and prevent NaNs
        for param in self.text_encoder.parameters():
            param.requires_grad = False

        # T5-Large dimension is 1024. We add 7 extra channels for our custom structural one-hot encodings
        in_channels = self.text_encoder.config.d_model + 7

        # Relational Layers to handle rule pathways distinctly
        self.conv1 = RGCNConv(in_channels, hidden_dim, num_relations=num_relations)
        self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations=num_relations)

        # Global attention pooling layer
        self.pool = AttentionalAggregation(gate_nn=nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        ))

        # Output layer for the 3 targets
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(hidden_dim, num_classes)
        )

    def compute_node_embeddings(self, node_texts, target_device):
        """Extracts frozen text feature states using Flan-T5-Large encoder to avoid NaN propagation."""
        if isinstance(node_texts, tuple):
            node_texts = list(node_texts)
        elif not isinstance(node_texts, list):
            node_texts = [str(node_texts)]

        # In case PyG nests lists during batching, flatten them
        flat_texts = []
        def flatten(lst):
            for item in lst:
                if isinstance(item, (list, tuple)):
                    flatten(item)
                else:
                    flat_texts.append(str(item))
        flatten(node_texts)

        inputs = self.tokenizer(flat_texts, padding=True, truncation=True, return_tensors="pt").to(target_device)
        with torch.no_grad():
            outputs = self.text_encoder(**inputs)
            node_features = outputs.last_hidden_state.mean(dim=1)
        return node_features

    def forward(self, node_texts, struct_tensor, edge_index, edge_type, batch):
        # Dynamically compute text features
        text_features = self.compute_node_embeddings(node_texts, edge_index.device)
        x_features = torch.cat([text_features, struct_tensor], dim=-1)

        h = self.conv1(x_features, edge_index, edge_type).relu()
        h = self.conv2(h, edge_index, edge_type).relu()
        graph_emb = self.pool(h, batch)
        return self.classifier(graph_emb)

## 2. AST Preprocessing with Rule-Based Contrapositive Injection

In [4]:
def augment_contrapositive(node):
    """
    Recursively searches for structural implication operators inside the AST dict.
    Injects logically equivalent contrapositive branches (not B -> not A) into children rows.
    """
    if not isinstance(node, dict):
        return node

    if "children" in node:
        augmented_children = []
        for child in node["children"]:
            augmented_children.append(augment_contrapositive(child))

            if child.get("type") == "keyword" and child.get("text") == "implies":
                try:
                    parent_children = node["children"]
                    idx = parent_children.index(child)
                    if idx > 0 and idx < len(parent_children) - 1:
                        antecedent = parent_children[idx - 1]
                        consequent = parent_children[idx + 1]

                        negated_consequent = {"type": "negation", "text": "not", "children": [consequent]}
                        negated_antecedent = {"type": "negation", "text": "not", "children": [antecedent]}
                        implies_keyword = {"type": "keyword", "text": "implies", "children": []}

                        augmented_children.append(negated_consequent)
                        augmented_children.append(implies_keyword)
                        augmented_children.append(negated_antecedent)
                except Exception:
                    pass
        node["children"] = augmented_children
    return node

def ast_to_syllogism_graph(ast_dict):
  """Transforms an AST into a directed NetworkX graph where all nodes

  sharing the same name/label (regardless of type, e.g., symbol 'x' and term 'x')
  are merged into a single unique node, accumulating all incoming and
  outgoing edges.
  """
  G = nx.DiGraph()

  # 1. Add Root Node
  root_node = "Syllogism"
  G.add_node(root_node, label="Syllogism", node_type="root")

  # Helper to safely locate statements in the AST structure
  def find_statements(node):
    stats = []
    if not isinstance(node, dict):
      return stats

    if node.get("type") == "stat":
      stats.append(node)

    for child in node.get("children", []):
      stats.extend(find_statements(child))

    return stats

  statements = find_statements(ast_dict)
  if not statements:
    return G

  premises = statements[:-1]
  conclusion = statements[-1]

  # Helper to split a term node's contents by commas (e.g., x,y or tecmo,japan)
  def extract_sub_terms_from_node(node):
    if not isinstance(node, dict):
      return []

    sub_terms = []
    current_letters = []
    for child in node.get("children", []):
      child_type = child.get("type")
      if child_type == "COMMA":
        if current_letters:
          sub_terms.append("".join(current_letters))
          current_letters = []
      elif child_type == "LETTER" and "text" in child:
        current_letters.append(child["text"])
      else:
        nested = extract_sub_terms_from_node(child)
        if nested:
          sub_terms.extend(nested)
    if current_letters:
      sub_terms.append("".join(current_letters))

    if not sub_terms and "text" in node:
      sub_terms.append(node["text"])

    return sub_terms

  # Helper to find text inside simple nodes like quantifier, keyword, symbol
  def get_node_text(node):
    if not isinstance(node, dict):
      return ""
    if "text" in node:
      return node["text"]
    for child in node.get("children", []):
      res = get_node_text(child)
      if res:
        return res
    return ""

  def collect_keywords(node, keywords):
    if not isinstance(node, dict):
      return
    if node.get("type") == "keyword":
      kw_text = get_node_text(node)
      if kw_text and kw_text not in keywords:
        keywords.append(kw_text)
    for child in node.get("children", []):
      collect_keywords(child, keywords)

  def find_atomic_propositions(node):
    aps = []
    if not isinstance(node, dict):
      return aps
    if node.get("type") == "atomicproposition":
      aps.append(node)
    for child in node.get("children", []):
      aps.extend(find_atomic_propositions(child))
    return aps

  # Helper to ensure unique nodes by label name (merges different types with same name)
  def add_merged_node(label, node_type):
    if not G.has_node(label):
      G.add_node(label, label=label, node_type=node_type)
    else:
      # Optional: update node_type to reflect multiple roles if needed
      existing_type = G.nodes[label].get("node_type", "")
      if node_type not in existing_type:
        G.nodes[label]["node_type"] = f"{existing_type},{node_type}"
    return label

  # Process a single statement (Premise or Conclusion)
  def process_statement(stat_node, parent_graph_node):
    quantifier_label = None
    symbol_label = None

    def find_quantifier_symbol(node):
      nonlocal quantifier_label, symbol_label
      if not isinstance(node, dict):
        return
      ntype = node.get("type")
      if ntype == "quantifier":
        q_text = get_node_text(node)
        if q_text:
          quantifier_label = add_merged_node(q_text, "quantifier")
          G.add_edge(parent_graph_node, quantifier_label)
      elif ntype == "symbol" and quantifier_label:
        s_text = get_node_text(node)
        if s_text:
          symbol_label = add_merged_node(s_text, "symbol")
          G.add_edge(quantifier_label, symbol_label)

      for child in node.get("children", []):
        find_quantifier_symbol(child)

    find_quantifier_symbol(stat_node)

    # Process Atomic Propositions (Predicate -> Argument Term Edges)
    aps = find_atomic_propositions(stat_node)
    statement_terms = set()

    for ap in aps:
      term_children = [
          c for c in ap.get("children", []) if c.get("type") == "term"
      ]
      ap_terms = []
      for tc in term_children:
        sub = extract_sub_terms_from_node(tc)
        ap_terms.extend(sub)

      if ap_terms:
        predicate_text = ap_terms[0]
        pred_label = add_merged_node(predicate_text, "term")
        G.add_edge(parent_graph_node, pred_label)
        statement_terms.add(predicate_text)

        for arg_text in ap_terms[1:]:
          arg_label = add_merged_node(arg_text, "term")
          G.add_edge(parent_graph_node, arg_label)
          statement_terms.add(arg_text)

          # Directed edge from predicate to argument term
          G.add_edge(pred_label, arg_label)

    # Fallback for any standalone terms not caught in atomic propositions
    def find_all_terms(node):
      if not isinstance(node, dict):
        return
      if node.get("type") == "term":
        for t in extract_sub_terms_from_node(node):
          if t not in statement_terms:
            t_label = add_merged_node(t, "term")
            G.add_edge(parent_graph_node, t_label)
            statement_terms.add(t)
      for child in node.get("children", []):
        find_all_terms(child)

    find_all_terms(stat_node)

    # Keywords -> Connect keyword to all terms in the statement
    keywords = []
    collect_keywords(stat_node, keywords)

    for kw in keywords:
      kw_label = add_merged_node(kw, "keyword")
      G.add_edge(parent_graph_node, kw_label)

      # Connect keyword to all terms present in this statement
      for t in statement_terms:
        G.add_edge(kw_label, t)

  # 2. Process Premises
  for idx, premise in enumerate(premises):
    premise_name = f"Premise_{idx + 1}"
    add_merged_node(premise_name, "premise")
    G.add_edge(root_node, premise_name)
    process_statement(premise, premise_name)

  # 3. Process Conclusion
  conclusion_name = "Conclusion"
  add_merged_node(conclusion_name, "conclusion")
  G.add_edge(root_node, conclusion_name)
  process_statement(conclusion, conclusion_name)

  return G

## 3. NetworkX to PyTorch Geometric Data Bridge Converter

In [5]:
NODE_TYPE_MAP = {"root": 0, "premise": 1, "conclusion": 2, "quantifier": 3, "symbol": 4, "term": 5, "keyword": 6}

def parse_string_to_multilabel(label_str):
    clean = str(label_str).strip().lower()
    if clean == "true": return [1.0, 0.0, 0.0]
    if clean == "false": return [0.0, 1.0, 0.0]
    return [0.0, 0.0, 1.0] # Unknown tracking mapping index

def convert_nx_to_pyg_data(G, label_vector, target_device):
    nodes_list = list(G.nodes())
    node_to_idx = {node: idx for idx, node in enumerate(nodes_list)}
    node_texts = [G.nodes[n].get("label", str(n)) for n in nodes_list]

    # Encode structural type context explicitly (7-dimensional one-hot)
    num_structural_types = len(NODE_TYPE_MAP)
    structural_features = []
    for n in nodes_list:
        n_type = G.nodes[n].get("node_type", "symbol")
        type_idx = NODE_TYPE_MAP.get(n_type, 4)
        one_hot = [0.0] * num_structural_types
        one_hot[type_idx] = 1.0
        structural_features.append(one_hot)

    struct_tensor = torch.tensor(structural_features, dtype=torch.float32, device=target_device)

    edge_sources, edge_targets, edge_types = [], [], []
    for u, v, data in G.edges(data=True):
        edge_sources.append(node_to_idx[u])
        edge_targets.append(node_to_idx[v])
        edge_types.append(data.get("relation", 0))

    edge_index = torch.tensor([edge_sources, edge_targets], dtype=torch.long)
    edge_attr = torch.tensor(edge_types, dtype=torch.long)
    y_tensor = torch.tensor([label_vector], dtype=torch.float)

    # Preserve raw node texts to process dynamically inside the forward pass
    return Data(node_texts=node_texts, struct_tensor=struct_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)

## 4. Ingestion Dataset Pipeline Builders

In [9]:
def build_pyg_split(df, target_device, run_augmentation=False):
    pyg_dataset = []
    for idx, row in df.iterrows():
        try:
            premises_str = str(row["Premises - CLIF"]).replace("'", '"')
            conclusion_str = str(row["Conclusions - CLIF"]).replace("'", '"')

            label_vector = parse_string_to_multilabel(row["label"])

            p_json = json.loads(premises_str)
            c_json = json.loads(conclusion_str)
            p_json["children"].append(c_json["children"][0]["children"][0])
            root_ast = p_json["children"][0]

            if run_augmentation:
                root_ast = augment_contrapositive(root_ast)

            nx_graph = ast_to_syllogism_graph(root_ast)
            pyg_data = convert_nx_to_pyg_data(nx_graph, label_vector, target_device)
            pyg_dataset.append(pyg_data)
        except Exception:
            continue
    return pyg_dataset

## 5. Training Loop with OVR Validation Threshold Optimization Suite

In [22]:
def train_model(model, train_loader, val_loader, epochs=5, accumulation_steps=8):
    # Using CrossEntropyLoss for mutually exclusive labels
    criterion = nn.CrossEntropyLoss()

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=0.01)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        optimizer.zero_grad()

        for i, data in enumerate(train_loader):
            data = data.to(device)

            out = model(data.node_texts, data.struct_tensor, data.edge_index, data.edge_attr, data.batch)
            # Extract target indices using argmax from one-hot style encoding
            target_indices = torch.argmax(data.y, dim=-1)
            raw_loss = criterion(out, target_indices)
            loss = raw_loss / accumulation_steps

            if torch.isnan(raw_loss):
                continue

            loss.backward()
            total_loss += raw_loss.item()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

        print(f"Epoch {epoch+1:02d} | Avg Epoch Loss: {total_loss/len(train_loader):.4f}")

## 6. Pipeline Execution Blueprint

In [23]:
import gc

# Free up GPU memory leftover from the crashed run
gc.collect()
torch.cuda.empty_cache()

def evaluate_test(model, test_loader, target_device):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for data in test_loader:
            data = data.to(target_device)
            logits = model(data.node_texts, data.struct_tensor, data.edge_index, data.edge_attr, data.batch)
            # Take softmax and argmax to produce mutually exclusive outputs
            probs = torch.softmax(logits, dim=-1)
            preds = torch.argmax(probs, dim=-1).cpu().numpy()
            targets = torch.argmax(data.y, dim=-1).cpu().numpy()

            all_preds.extend(preds)
            all_targets.extend(targets)

    return f1_score(all_targets, all_preds, average='macro', zero_division=0)

# Instantiate GNN
gnn_model = LogicGraphClassifier(pretrained_model_name="google/flan-t5-large", num_relations=2).to(device)

# Load data partitions
train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/folio_ast_train.csv")
val_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/folio_ast_val.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/folio_ast_test.csv")

print("Processing raw structural graphs (Contrapositive Augmentation Disabled)...")
train_dataset = build_pyg_split(train_df, device, run_augmentation=False)
val_dataset = build_pyg_split(val_df, device, run_augmentation=False)
test_dataset = build_pyg_split(test_df, device, run_augmentation=False)

# Reduce batch size to avoid out-of-memory errors
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

train_model(gnn_model, train_loader, val_loader, epochs=5, accumulation_steps=8)

test_f1 = evaluate_test(gnn_model, test_loader, device)
print(f"\nFinal Target Test Split Macro-F1 Performance (Softmax): {test_f1*100:.2f}%")

Loading weights:   0%|          | 0/557 [00:00<?, ?it/s]

[transformers] T5Model LOAD REPORT from: google/flan-t5-large
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing raw structural graphs (Contrapositive Augmentation Disabled)...


/usr/local/lib/python3.13/dist-packages/torch_geometric/data/data.py:187: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y', 'edge_attr', 'node_texts', 'struct_tensor', 'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  return sum([v.num_nodes for v in self.node_stores])
/usr/local/lib/python3.13/dist-packages/torch_geometric/data/collate.py:142: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y', 'edge_attr', 'node_texts', 'struct_tensor', 'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  repeats = [store.num_nodes or 0 for store in stores]


Epoch 01 | Avg Epoch Loss: 1.0945
Epoch 02 | Avg Epoch Loss: 1.0918
Epoch 03 | Avg Epoch Loss: 1.0890
Epoch 04 | Avg Epoch Loss: 1.0889
Epoch 05 | Avg Epoch Loss: 1.0890

Final Target Test Split Macro-F1 Performance (Softmax): 17.58%


In [24]:
from sklearn.metrics import confusion_matrix, classification_report

def analyze_prediction_errors_softmax(model, test_loader, target_device):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for data in test_loader:
            data = data.to(target_device)
            logits = model(data.node_texts, data.struct_tensor, data.edge_index, data.edge_attr, data.batch)
            probs = torch.softmax(logits, dim=-1)
            preds = torch.argmax(probs, dim=-1).cpu().numpy()
            targets = torch.argmax(data.y, dim=-1).cpu().numpy()

            all_preds.extend(preds)
            all_targets.extend(targets)

    target_names = ['True', 'False', 'Unknown']

    # Generate a single confusion matrix representing mutually exclusive classes
    cm = confusion_matrix(all_targets, all_preds)

    print("=== Mutually Exclusive Confusion Matrix ===\n")
    print("             Predicted")
    print("             True  False  Unknown")
    for i, name in enumerate(target_names):
        print(f"Actual {name:7s} {cm[i, 0]:4d} {cm[i, 1]:5d} {cm[i, 2]:8d}")
    print("-" * 45)

    print("\n=== Classification Report ===\n")
    print(classification_report(all_targets, all_preds, target_names=target_names, zero_division=0))

analyze_prediction_errors_softmax(gnn_model, test_loader, device)

/usr/local/lib/python3.13/dist-packages/torch_geometric/data/data.py:187: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y', 'edge_attr', 'node_texts', 'struct_tensor', 'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  return sum([v.num_nodes for v in self.node_stores])
/usr/local/lib/python3.13/dist-packages/torch_geometric/data/collate.py:142: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'y', 'edge_attr', 'node_texts', 'struct_tensor', 'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  repeats = [store.num_nodes or 0 for store in stores]


=== Mutually Exclusive Confusion Matrix ===

             Predicted
             True  False  Unknown
Actual True      72     0        0
Actual False     64     0        0
Actual Unknown   65     0        0
---------------------------------------------

=== Classification Report ===

              precision    recall  f1-score   support

        True       0.36      1.00      0.53        72
       False       0.00      0.00      0.00        64
     Unknown       0.00      0.00      0.00        65

    accuracy                           0.36       201
   macro avg       0.12      0.33      0.18       201
weighted avg       0.13      0.36      0.19       201

